# 4.3 Rebuild the Module 2 Agent as a Graph

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

**Prerequisites:** 2.3 (the hand-rolled ReAct loop and `create_agent`), 2.4 (middleware), 4.2 (state, nodes, edges, conditional edges). You need an `OPENAI_API_KEY` (lesson 0.2), or an Ollama model for the local path at the end.

## Learning objectives

- Rebuild the Module 2 ReAct agent as an explicit LangGraph graph: an agent node, a `ToolNode`, and a `tools_condition` router.
- Explain what `create_agent` wraps around that skeleton: middleware, prompts, and runtime conveniences.
- Insert a custom step into the loop (a gate between the agent and its tools) that the prebuilt harness cannot express as cleanly.
- Decide, for a given task, when to stay with `create_agent` and when to build the graph by hand.

## What we already have: two versions of the same loop

Lesson 2.3 left you with two implementations of ReAct. The hand-rolled loop: a system prompt, a model with tools bound via `bind_tools`, and a `for` loop that appends the model's reply, runs any tool calls, appends the results, and repeats. And `create_agent`: one call that builds the same loop for you, returning a compiled LangGraph graph with middleware slots.

The second one is secretly the third thing you are about to build. `create_agent` does not implement a different loop; it builds a graph. This lesson opens the hood and rebuilds that graph by hand, so you can see exactly which parts are essential and which parts are conveniences the harness adds. Once you have built it, you can do things the harness makes awkward: route on more than "did the model call a tool", insert nodes between the model and the tools, pause before a tool runs.

## The three moving parts

A ReAct agent, drawn as a graph, has exactly three moving parts:

1. **An agent node** that calls the model with tools bound and appends the reply to the message list.
2. **A tools node** that executes the tool calls from the last message and appends the results.
3. **A router** that looks at the last message: if it contains tool calls, go to the tools node; otherwise the run is done.

The loop is the edges: agent to tools, tools back to agent, agent to `END`. Lesson 4.2 taught you every mechanic here: `MessagesState` for the message list with its reducer already attached, plain functions as nodes, a conditional edge with a router. LangGraph even ships the tools node and the router prebuilt, at `langgraph.prebuilt`:

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

`ToolNode(tools)` reads the tool calls from the last `AIMessage` in state, runs each one, and returns the `ToolMessage` results. `tools_condition` returns `"tools"` when the last message has tool calls and `END` otherwise. Both were verified by install on the pinned langgraph 1.2.16.

## The full rebuild

The same tools as lesson 2.3, the same system prompt, wired as a graph:

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

@tool
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression.

    Use this for any math instead of doing mental arithmetic.
    Args:
        expression: A Python arithmetic expression, e.g. '38 + 47'.
    """
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception:
        return f"Error: could not evaluate '{expression}'. Send a plain arithmetic expression."

@tool
def get_current_time() -> str:
    """Return the current UTC date and time as an ISO string.

    Use this whenever the user asks about 'now', 'today', or the current date.
    """
    from datetime import datetime, timezone
    return datetime.now(timezone.utc).isoformat()

tools = [calculate, get_current_time]
llm_with_tools = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools)

def agent_node(state: MessagesState) -> dict:
    reply = llm_with_tools.invoke(state["messages"])
    return {"messages": [reply]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

graph = builder.compile()

result = graph.invoke({
    "messages": [
        SystemMessage(
            "You are a helpful assistant. When you need a calculation, use the "
            "calculate tool instead of doing arithmetic yourself. When you need the "
            "current date or time, use get_current_time. Answer only when you have "
            "everything you need."
        ),
        HumanMessage("What is 38 + 47? And what is today's date?"),
    ]
})
print(result["messages"][-1].content)

Run it and you get the same answer the 2.3 loop and the `create_agent` version give. Three details to notice:

- `agent_node` is a thin wrapper around one model call. The ReAct loop is no longer a `for` loop in your code; it is the edge from `tools` back to `agent`, plus the router deciding when to stop. The iteration logic moved from code you wrote into wiring you can see.
- `MessagesState` carries the `add_messages` reducer, so `{"messages": [reply]}` appends instead of replacing. Lesson 4.2's default-replacement warning applies here too: if you ever declare the message list without the reducer, the conversation silently collapses to the last message.
- `tools_condition` reads only the last message. The mapping `{"tools": "tools", END: END}` is the complete routing logic. There is nothing else to the ReAct pattern at the graph level.

Ollama path: replace the model line with `from langchain_ollama import ChatOllama` and `ChatOllama(model="llama3.1:8b").bind_tools(tools)`. Tool calling works on the pinned langchain-ollama 1.1.0; smaller models route less reliably, which is itself a useful observation about model capability versus harness cleverness.

## Three implementations, side by side

| | Hand-rolled loop (2.3) | Explicit graph (this lesson) | `create_agent` (2.3/2.4) |
|---|---|---|---|
| ReAct logic | a `for` loop you wrote | visible edges and one router | inside the harness |
| Loop budget | `range(6)` by hand | `recursion_limit` config, or a counter in state you route on | `ModelCallLimitMiddleware` |
| Tool errors | your try/except | your node code | `ToolErrorMiddleware`, `ToolRetryMiddleware` |
| Streaming | hand-rolled prints | `stream()` modes (lesson 4.4) | `stream()` modes, same runtime |
| Pause before a tool runs | rewrite the loop | `interrupt_before=["tools"]` on compile | `interrupt_before` or `HumanInTheLoopMiddleware` |
| Memory across turns | your list | a checkpointer on `compile` | a checkpointer on `compile` |

The row that matters is the first one. The hand-rolled loop and the explicit graph are the same program written in two notations; `create_agent` is the explicit graph with a harness around it. When `create_agent` misbehaves, this is the skeleton you picture: one model node, one tools node, one router.

## What the harness adds

If the graph is the skeleton, `create_agent` adds the parts you do not want to rewrite every time:

- **Middleware** (lesson 2.4): error conversion, retries with backoff, call budgets, summarisation, human-in-the-loop approvals. You can add equivalent behaviour to the explicit graph, but it is node code you write and test yourself, while the middleware is maintained by the LangChain team.
- **Prompt management**: `system_prompt=` is threaded into the agent node for you, with the right message placement and update behaviour.
- **Structured output**: `response_format=` gives the agent a typed final answer, a pattern you would otherwise hand-roll with a second model call.
- **Dynamic models**: passing `model="openai:gpt-4o-mini"` as a string builds the chat model for you, which keeps configuration in one place.

None of these change the loop. They change how much of the loop's surroundings you maintain.

## What the explicit graph buys you

The reason to build the graph by hand is control over the wiring. Two examples that are awkward inside `create_agent`:

**A gate between the agent and its tools.** Insert a node on the tools path that inspects the pending tool calls and decides whether they may run. The wiring is one extra node and two edges:

In [ ]:
builder.add_node("gate", gate_node)          # inspects state, approves or blocks
builder.add_edge("agent", "gate")
builder.add_conditional_edges("gate", gate_router, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

`gate_node` reads the last message's `tool_calls` and returns a verdict into state; `gate_router` routes on it. Inside `create_agent` you would express this as `HumanInTheLoopMiddleware` or a custom middleware, which is the right choice when the gate is generic, but a node is clearer when the gate has domain logic (block refunds above a threshold, require a ticket id on every search call).

**A router that reads more than tool calls.** `tools_condition` looks only at the last message. Your own router can read anything in state: a retry counter, a cost accumulator, a flag the tools node set. "Stop looping when the model has called search three times" is a conditional edge on a counter, not on the message shape. That is the difference between a loop you control and a loop you configure.

The honest rule of thumb from production, repeated from lesson 2.3 because it now has teeth: start with `create_agent`, and drop to the explicit graph when you can point at one specific behaviour the harness cannot express. "I want to see the loop" is a learning reason, and it is a good one; it is not a production reason.

## A preview: pausing before the tools run

One line on the explicit graph buys the human-in-the-loop pattern that lesson 5.2 covers in depth:

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

graph = builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["tools"],
)

With a checkpointer attached and `interrupt_before=["tools"]`, the run pauses every time the router sends work to the tools node, and resumes when you approve. You will use exactly this shape in the lesson 4.5 lab. The mechanics of pausing and resuming are lesson 4.4.

## What can go wrong

- **ToolNode with a tool list that does not match the bound tools.** If the model calls a tool that is not in `ToolNode`'s list, the tools node raises instead of returning an observation. Keep one `tools` list, bind it to the model and pass it to `ToolNode`; two lists drift.
- **A router that only checks `tool_calls` when you need more.** `tools_condition` is correct for textbook ReAct and wrong the moment your stopping rule involves a counter, a cost budget, or a flag. The symptom is a loop that only stops at `recursion_limit`; the fix is your own router reading your own state.
- **Assuming the graph adds reliability by itself.** The explicit graph has no retries, no error conversion, no call budget unless you add them. `create_agent`'s middleware is the difference between the two columns in the table, and rebuilding the graph is the moment that difference becomes visible instead of theoretical.
- **Forgetting the tools-to-agent edge.** `builder.add_edge("tools", "agent")` is the loop. Without it, the first tool call runs and the run ends at the tools node with no answer, because `ToolNode` has no outgoing edge and a node with no way out fails. Say the loop with an edge and mean it.
- **Reaching for the explicit graph out of habit.** Every custom graph is code you maintain through LangGraph upgrades. The harness absorbs API churn; your hand-built wiring does not. Build by hand when the wiring is the feature, not by default.

## Check your understanding

**1. Draw the ReAct agent as a graph in words: which nodes, which edges, and where is the loop?**
Two nodes: `agent` (calls the model with tools bound, appends the reply) and `tools` (a `ToolNode` that executes the pending tool calls). Edges: `START` to `agent`; a conditional edge from `agent` via `tools_condition` to `tools` when the last message has tool calls, else to `END`; and a fixed edge from `tools` back to `agent`. The loop is the tools-to-agent edge plus the router: each pass appends a model reply and tool results until a reply has no tool calls, and the router sends that reply to `END`.

**2. `create_agent` returns a compiled LangGraph graph. Name two things the harness adds on top of the three-piece skeleton from this lesson.**
Any two of: middleware (error handling, retries, call budgets, summarisation, human-in-the-loop approvals), prompt management (`system_prompt=` threaded into the agent node), structured final answers via `response_format=`, dynamic model construction from a `"provider:model"` string, and maintained compatibility across LangChain upgrades.

**3. Your explicit graph runs the tools node once and then stops, with no final answer. What is the most likely wiring mistake?**
The edge from `tools` back to `agent` is missing, so after the first tool execution the run has nowhere to go. A node with no outgoing edge is a dead end: add `builder.add_edge("tools", "agent")` to close the loop.

**4. You want the agent to stop after three search-tool calls even if the model wants to keep searching. Where does that logic live in each of the three implementations?**
Hand-rolled loop: a counter in the `for` loop, checked before each model call. Explicit graph: a counter in state, incremented by the tools node, read by your own router on the conditional edge out of `agent` (or out of the tools node). `create_agent`: `ToolCallLimitMiddleware` scoped to the search tool with `run_limit=3`.

**5. When should you build the graph by hand instead of using `create_agent`?**
When you can name one specific behaviour the harness cannot express: a router that reads more than the last message's tool calls, a custom node between the agent and the tools (a domain-specific gate), or a topology that is not the standard loop at all. Start with `create_agent` otherwise; hand-built wiring is maintenance you own through every upgrade.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.